# AVeriTeC Claim Verification (Real-World, Hard Benchmark)

Runs the proposed groundedness pipeline on AVeriTeC (NeurIPS 2023): 60 sampled claims with QA evidence.

In [ ]:
import os, json, time, re, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix
import faiss
import ollama
from huggingface_hub import hf_hub_download

sns.set_theme(style="whitegrid")
GEN_MODEL="gemma4:e4b"; EMB_MODEL="nomic-embed-text:latest"; SEED=42
random.seed(SEED); np.random.seed(SEED)


In [ ]:
def llm(prompt, temperature=0.1, num_predict=16):
    r = ollama.chat(model=GEN_MODEL, messages=[{"role":"user","content":prompt}],
                    options={"temperature":temperature,"num_predict":num_predict}, think=False)
    return r["message"]["content"].strip()

def embed_texts(texts, batch=16):
    out=[]
    for i in range(0,len(texts),batch):
        r=ollama.embed(model=EMB_MODEL, input=texts[i:i+batch])
        out.extend(r["embeddings"])
    return np.array(out,dtype=np.float32)


In [ ]:
p = hf_hub_download(repo_id="chenxwh/AVeriTeC", filename="data/dev.json")
data = json.load(open(p, encoding="utf-8"))
supported=[x for x in data if x["label"]=="Supported"]
refuted=[x for x in data if x["label"]=="Refuted"]
unver=[x for x in data if x["label"] in ("Not Enough Evidence","Conflicting Evidence/Cherrypicking")]
random.seed(SEED)
samp=random.sample(supported,20)+random.sample(refuted,20)+random.sample(unver,20)

claims=[]; docs=[]; amap={}
def add_doc(t):
    if t not in amap:
        amap[t]=f"AV-{len(amap)}"; docs.append({"doc_id":amap[t],"text":t})

for x in samp:
    lab = "true" if x["label"]=="Supported" else ("false" if x["label"]=="Refuted" else "unverifiable")
    for q in x["questions"]:
        for a in q["answers"]:
            add_doc(a["answer"])
    claims.append({"id":x["claim"][:8],"claim":x["claim"],"label":lab})
df=pd.DataFrame(claims)
print(df.label.value_counts())


In [ ]:
# build index
corpus=pd.DataFrame(docs)
emb=embed_texts(corpus.text.tolist())
faiss.normalize_L2(emb)
index=faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
print("docs", len(corpus))


In [ ]:
def retrieve(q,k=5):
    qe=embed_texts([q]); faiss.normalize_L2(qe)
    sc,ids=index.search(qe,k)
    return [corpus.iloc[i].text for i in ids[0] if i>=0]

def grounding_score(claim):
    res=retrieve(claim,k=5)
    ctx="\n".join(res)
    prompt="Evidence:\n"+ctx+"\n\nClaim: "+claim+"\nDoes the evidence support the claim? Answer SUPPORTED, CONTRADICTED, or INSUFFICIENT."
    out=llm(prompt,num_predict=16)
    t=out.upper()
    if "CONTRADICT" in t: return 0.0,res
    if "SUPPORT" in t: return 1.0,res
    return 0.5,res

def score_claim(claim):
    D,res=grounding_score(claim)
    # no structured records/policy for AVeriTeC
    F=0.5; P=0.5
    G=0.4*F+0.3*P+0.3*D
    if D>=0.9: dec="PASS"
    elif D<=0.1: dec="BLOCK"
    else: dec="FLAG"
    return {"F":F,"P":P,"D":D,"G":G,"decision":dec,"pred":{"PASS":"true","BLOCK":"false","FLAG":"unverifiable"}[dec]}

details=[]
for _,r in df.iterrows():
    d=score_claim(r.claim); d["id"]=r.id; d["label"]=r.label; d["claim"]=r.claim
    details.append(d)
ddf=pd.DataFrame(details)
print(ddf[["id","label","D","decision","pred"]].to_string(index=False))
print("accuracy", accuracy_score(df.label, ddf.pred))


In [ ]:
# confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm=confusion_matrix(df.label, ddf.pred, labels=["true","false","unverifiable"])
ConfusionMatrixDisplay(cm, display_labels=["true","false","unverifiable"]).plot(cmap="Blues")
plt.title("AVeriTeC proposed pipeline"); plt.show()


## Notes
- AVeriTeC is a hard, real-world benchmark; our simple retrieval+entailment pipeline is a baseline.
- For higher accuracy, decompose claims into the dataset's question-answer pairs (AVeriTeC's intended approach).